In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import IFrame, display

URL = "https://huggingface.co/datasets/Vidulaae/sales-analysis/resolve/main/Sample%20-%20Superstore.csv"
df = pd.read_csv(URL, encoding="latin-1")
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df = df.dropna(subset=["Order Date"])
df["Year"] = df["Order Date"].dt.year

BG, GRID = "#0E1117", "#222833"
GREEN, RED, BLUE, GOLD, PURPLE = "#2EC4B6", "#FF5D73", "#00A6FB", "#F5B700", "#B388EB"

yearly = df.groupby("Year").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
                                Orders=("Order ID", "nunique"))
yearly["Margin"] = yearly["Profit"] / yearly["Sales"] * 100
cur, prev = yearly.iloc[-1], yearly.iloc[-2]
cur_year = yearly.index[-1]

monthly = df.set_index("Order Date").resample("MS").agg({"Sales": "sum", "Profit": "sum"})
cat_sales = df.groupby("Category")["Sales"].sum()
sub = df.groupby("Sub-Category")["Profit"].sum().sort_values()

bins = [-0.001, 0, 0.1, 0.2, 0.3, 0.5, 1]
labels = ["0%", "1-10%", "11-20%", "21-30%", "31-50%", "50%+"]
df["Discount Band"] = pd.cut(df["Discount"], bins=bins, labels=labels)
disc = df.groupby("Discount Band", observed=True).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
disc["Margin"] = disc["Profit"] / disc["Sales"] * 100

states = df.groupby("State")["Profit"].sum().sort_values(ascending=False).head(10).sort_values()

heat = df.pivot_table(index="Segment", columns="Region", values=["Profit", "Sales"], aggfunc="sum")
margin_heat = (heat["Profit"] / heat["Sales"] * 100).round(1)

fig = make_subplots(
    rows=4, cols=4,
    specs=[
        [{"type": "indicator"}] * 4,
        [{"colspan": 3, "secondary_y": True}, None, None, {"type": "domain"}],
        [{"colspan": 2}, None, {"colspan": 2}, None],
        [{"colspan": 2}, None, {"colspan": 2}, None],
    ],
    row_heights=[0.12, 0.26, 0.31, 0.31],
    vertical_spacing=0.075, horizontal_spacing=0.07,
    subplot_titles=["", "", "", "",
                    "Monthly sales and profit", "Sales mix by category",
                    "Profit by sub-category", "Margin by discount band",
                    "Top 10 states by profit", "Margin by segment and region"],
)

kpis = [
    ("Sales", cur["Sales"], prev["Sales"], True, "$"),
    ("Profit", cur["Profit"], prev["Profit"], True, "$"),
    ("Profit Margin", cur["Margin"], prev["Margin"], False, ""),
    ("Orders", cur["Orders"], prev["Orders"], True, ""),
]
for i, (name, val, ref, rel, pre) in enumerate(kpis, start=1):
    fig.add_trace(go.Indicator(
        mode="number+delta", value=val,
        number=dict(prefix=pre, suffix="%" if name == "Profit Margin" else "",
                    valueformat=".1f" if name == "Profit Margin" else ",.0f",
                    font=dict(size=40, color="white")),
        delta=dict(reference=ref, relative=rel,
                   valueformat=".1%" if rel else ".1f",
                   suffix="" if rel else " pts",
                   increasing=dict(color=GREEN), decreasing=dict(color=RED)),
        title=dict(text=f"<b>{name}</b><br><span style='font-size:12px;color:#9aa0a6'>{cur_year} vs {cur_year - 1}</span>"),
    ), row=1, col=i)

fig.add_trace(go.Bar(x=monthly.index, y=monthly["Sales"], name="Sales",
                     marker_color=BLUE, opacity=0.55), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=monthly.index, y=monthly["Profit"], name="Profit",
                         line=dict(color=GOLD, width=3), mode="lines+markers"),
              row=2, col=1, secondary_y=True)

fig.add_trace(go.Pie(labels=cat_sales.index, values=cat_sales.values, hole=0.62,
                     marker=dict(colors=[BLUE, GOLD, PURPLE], line=dict(color=BG, width=3)),
                     textinfo="percent", sort=False, showlegend=False), row=2, col=4)

fig.add_trace(go.Bar(x=sub.values, y=sub.index, orientation="h",
                     marker_color=[GREEN if v > 0 else RED for v in sub.values],
                     text=[f"${v:,.0f}" for v in sub.values], textposition="outside",
                     showlegend=False), row=3, col=1)

fig.add_trace(go.Bar(x=disc.index.astype(str), y=disc["Margin"],
                     marker_color=[GREEN if v > 0 else RED for v in disc["Margin"]],
                     text=[f"{v:.1f}%" for v in disc["Margin"]], textposition="outside",
                     showlegend=False), row=3, col=3)

fig.add_trace(go.Bar(x=states.values, y=states.index, orientation="h",
                     marker_color=GREEN, text=[f"${v:,.0f}" for v in states.values],
                     textposition="outside", showlegend=False), row=4, col=1)

fig.add_trace(go.Heatmap(z=margin_heat.values, x=margin_heat.columns, y=margin_heat.index,
                         colorscale=[[0, RED], [0.5, "#1b2029"], [1, GREEN]], zmid=0,
                         texttemplate="%{z:.1f}%", showscale=False,
                         xgap=3, ygap=3), row=4, col=3)

fig.update_layout(
    template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family="Inter, Segoe UI, Arial", size=13),
    autosize=True, height=1500, margin=dict(t=110, l=60, r=40, b=40),
    title=dict(text="<b>Executive Dashboard: Superstore Performance</b><br>"
                    "<sup style='color:#9aa0a6'>Sales, profitability and discount impact across categories, regions and customer segments</sup>",
               x=0.02, xanchor="left"),
    legend=dict(orientation="h", x=0.0, y=0.715, bgcolor="rgba(0,0,0,0)"),
    bargap=0.25,
)
fig.update_xaxes(showgrid=False, zeroline=False)
fig.update_yaxes(gridcolor=GRID, zeroline=False)
fig.update_yaxes(title_text="Sales ($)", secondary_y=False, row=2, col=1)
fig.update_yaxes(title_text="Profit ($)", secondary_y=True, showgrid=False, row=2, col=1)
fig.update_yaxes(showgrid=False, row=3, col=1)
fig.update_yaxes(showgrid=False, row=4, col=1)
fig.update_xaxes(title_text="Total profit ($)", row=3, col=1)
fig.update_yaxes(title_text="Profit margin (%)", row=3, col=3)
fig.update_xaxes(title_text="Total profit ($)", row=4, col=1)
fig.update_annotations(font=dict(size=14, color="#c9d1d9"))

fig.write_html("executive_dashboard.html", include_plotlyjs="cdn", full_html=True,
               config={"responsive": True, "displaylogo": False})

display(IFrame("executive_dashboard.html", width="100%", height=1520))

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

URL = "https://huggingface.co/datasets/Vidulaae/sales-analysis/resolve/main/Sample%20-%20Superstore.csv"
df = pd.read_csv(URL, encoding="latin-1")
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df = df.dropna(subset=["Order Date"])
df["Year"] = df["Order Date"].dt.year

BG, GRID = "#0E1117", "#222833"
GREEN, RED, BLUE, GOLD, PURPLE = "#2EC4B6", "#FF5D73", "#00A6FB", "#F5B700", "#B388EB"

yearly = df.groupby("Year").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
                                Orders=("Order ID", "nunique"))
yearly["Margin"] = yearly["Profit"] / yearly["Sales"] * 100
cur, prev = yearly.iloc[-1], yearly.iloc[-2]
cur_year = yearly.index[-1]

monthly = df.set_index("Order Date").resample("MS").agg({"Sales": "sum", "Profit": "sum"})
cat_sales = df.groupby("Category")["Sales"].sum()
sub = df.groupby("Sub-Category")["Profit"].sum().sort_values()

bins = [-0.001, 0, 0.1, 0.2, 0.3, 0.5, 1]
labels = ["0%", "1-10%", "11-20%", "21-30%", "31-50%", "50%+"]
df["Discount Band"] = pd.cut(df["Discount"], bins=bins, labels=labels)
disc = df.groupby("Discount Band", observed=True).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
disc["Margin"] = disc["Profit"] / disc["Sales"] * 100

fig = make_subplots(
    rows=3, cols=4,
    specs=[
        [{"type": "indicator"}] * 4,
        [{"colspan": 3, "secondary_y": True}, None, None, {"type": "domain"}],
        [{"colspan": 2}, None, {"colspan": 2}, None],
    ],
    row_heights=[0.13, 0.32, 0.55],
    vertical_spacing=0.075, horizontal_spacing=0.07,
    subplot_titles=["", "", "", "",
                    "Monthly sales and profit", "Sales mix by category",
                    "Profit by sub-category", "Margin by discount band"],
)

kpis = [
    ("Sales", cur["Sales"], prev["Sales"], True, "$"),
    ("Profit", cur["Profit"], prev["Profit"], True, "$"),
    ("Profit Margin", cur["Margin"], prev["Margin"], False, ""),
    ("Orders", cur["Orders"], prev["Orders"], True, ""),
]
for i, (name, val, ref, rel, pre) in enumerate(kpis, start=1):
    fig.add_trace(go.Indicator(
        mode="number+delta", value=val,
        number=dict(prefix=pre, suffix="%" if name == "Profit Margin" else "",
                    valueformat=".1f" if name == "Profit Margin" else ",.0f",
                    font=dict(size=40, color="white")),
        delta=dict(reference=ref, relative=rel,
                   valueformat=".1%" if rel else ".1f",
                   suffix="" if rel else " pts",
                   increasing=dict(color=GREEN), decreasing=dict(color=RED)),
        title=dict(text=f"<b>{name}</b><br><span style='font-size:12px;color:#9aa0a6'>{cur_year} vs {cur_year - 1}</span>"),
    ), row=1, col=i)

fig.add_trace(go.Bar(x=monthly.index, y=monthly["Sales"], name="Sales",
                     marker_color=BLUE, opacity=0.55), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=monthly.index, y=monthly["Profit"], name="Profit",
                         line=dict(color=GOLD, width=3), mode="lines+markers"),
              row=2, col=1, secondary_y=True)

fig.add_trace(go.Pie(labels=cat_sales.index, values=cat_sales.values, hole=0.62,
                     marker=dict(colors=[BLUE, GOLD, PURPLE], line=dict(color=BG, width=3)),
                     textinfo="percent", sort=False, showlegend=False), row=2, col=4)

fig.add_trace(go.Bar(x=sub.values, y=sub.index, orientation="h",
                     marker_color=[GREEN if v > 0 else RED for v in sub.values],
                     text=[f"${v:,.0f}" for v in sub.values], textposition="outside",
                     showlegend=False), row=3, col=1)

fig.add_trace(go.Bar(x=disc.index.astype(str), y=disc["Margin"],
                     marker_color=[GREEN if v > 0 else RED for v in disc["Margin"]],
                     text=[f"{v:.1f}%" for v in disc["Margin"]], textposition="outside",
                     showlegend=False), row=3, col=3)

fig.update_layout(
    template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family="Inter, Segoe UI, Arial", size=13),
    autosize=True, height=1150, margin=dict(t=110, l=60, r=40, b=40),
    title=dict(text="<b>Executive Dashboard: Superstore Performance</b><br>"
                    "<sup style='color:#9aa0a6'>Sales, profitability and discount impact across categories, regions and customer segments</sup>",
               x=0.02, xanchor="left"),
    legend=dict(orientation="h", x=0.0, y=0.84, yanchor="bottom", bgcolor="rgba(0,0,0,0)"),
    bargap=0.25,
)
fig.update_xaxes(showgrid=False, zeroline=False)
fig.update_yaxes(gridcolor=GRID, zeroline=False)
fig.update_yaxes(title_text="Sales ($)", secondary_y=False, row=2, col=1)
fig.update_yaxes(title_text="Profit ($)", secondary_y=True, showgrid=False, row=2, col=1)
fig.update_yaxes(showgrid=False, row=3, col=1)
fig.update_xaxes(title_text="Total profit ($)", row=3, col=1)
fig.update_yaxes(title_text="Profit margin (%)", row=3, col=3)
fig.update_annotations(font=dict(size=14, color="#c9d1d9"))

fig.show(config={"responsive": True, "displaylogo": False})